# Fase 0 — Setup do ambiente

Preparação do ambiente de execução do estudo comparativo **U-Net (CNN) vs SegFormer (ViT)** para segmentação semântica de cafezais.

Este estágio detecta a plataforma de execução, instala as dependências necessárias, carrega a configuração única do projeto, fixa as sementes, autentica o Google Earth Engine e registra o ambiente do run. As saídas são: ambiente pronto, configuração carregada e credenciais do GEE validadas.

## Obtenção do repositório

Garante a presença do pacote `src/` numa área gravável da plataforma: no Colab clona o repositório público para `/content`; no Kaggle copia o dataset somente-leitura de `/kaggle/input` para `/kaggle/working` (ou clona do GitHub se não houver dataset). No ambiente local a etapa é ignorada, pois o repositório já está no diretório corrente.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
from pathlib import Path


# URL pública do repositório, usada como fallback quando não há dataset montado.
REPO_URL = "https://github.com/jotap1101/tcc.git"
REPO_NAME = "tcc"


def _copy_repo(source: Path, dest: Path) -> None:
    """Copia o repositório para a área gravável, ignorando metadados de versionamento."""
    dest.mkdir(parents=True, exist_ok=True)
    for item in source.iterdir():
        if item.name in {".git", "__pycache__", ".ruff_cache", ".mypy_cache", ".pytest_cache"}:
            continue
        target = dest / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)


# Detecta Colab com segurança, mesmo quando o pacote google não existe.
is_colab = "COLAB_GPU" in os.environ
if not is_colab:
    try:
        is_colab = importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        is_colab = False

# Resolve a área gravável e a fonte do repositório conforme a plataforma.
if is_colab:
    working_dir = Path("/content") / REPO_NAME
    repo_source = None
elif Path("/kaggle").is_dir():
    working_dir = Path("/kaggle/working") / REPO_NAME
    repo_source = next(
        (p for p in Path("/kaggle/input").glob("*") if (p / "src" / "config.yaml").is_file()),
        None,
    )
else:
    working_dir = None
    repo_source = None

# Garante o repositório na área gravável: copia o dataset do Kaggle ou clona do GitHub.
if working_dir is not None:
    if (working_dir / "src" / "config.yaml").is_file():
        print(f"Repositório já presente em {working_dir}.")
    elif repo_source is not None:
        _copy_repo(repo_source, working_dir)
        print(f"Repositório copiado de {repo_source} para {working_dir}.")
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(working_dir)], check=True)
        print(f"Repositório clonado em {working_dir}.")
    os.environ["TCC_ROOT"] = str(working_dir)
else:
    print("Ambiente local: repositório já disponível no diretório corrente.")

## Detecção da raiz do repositório

Localiza a raiz do repositório pelo marcador `src/config.yaml` nos diretórios corrente, ancestrais e raízes de montagem das plataformas de nuvem (Kaggle/Colab), além do override via variável de ambiente `TCC_ROOT`. A raiz é inserida no caminho de importação, garantindo o acesso ao pacote `src/`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


# Raízes de busca: env TCC_ROOT, diretório corrente com ancestrais e montagens
# de repositórios nas plataformas de nuvem (Kaggle /kaggle/input, Colab /content).
def _candidate_bases() -> list[Path]:
    bases: list[Path] = []
    tcc_root = os.environ.get("TCC_ROOT")
    if tcc_root:
        bases.append(Path(tcc_root))
    bases.extend([Path.cwd(), *Path.cwd().parents])
    for mount in (Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")):
        if mount.is_dir():
            bases.append(mount)
    return bases


# Verifica a base e seus subdiretórios imediatos em busca do marcador da raiz.
def _find_project_root() -> Path:
    for base in _candidate_bases():
        for candidate in [base, *base.glob("*")]:
            if candidate.is_dir() and (candidate / "src" / "config.yaml").is_file():
                return candidate
    raise RuntimeError("Raiz do repositório não localizada (src/config.yaml ausente).")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Raiz do projeto: {PROJECT_ROOT}")

## Detecção da plataforma

Identifica o ambiente de execução (Kaggle, Colab ou local) para adaptar a instalação de dependências e a leitura de segredos.

In [ ]:
import importlib.util
import os


def _is_colab_runtime() -> bool:
    """Detecta Colab com segurança, mesmo quando o pacote google não existe."""
    if "COLAB_GPU" in os.environ:
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


# Colab é detectado primeiro, pois /kaggle também existe nos runtimes do Colab.
def detect_platform() -> str:
    if _is_colab_runtime():
        return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir():
        return "kaggle"
    return "local"


PLATFORM = detect_platform()
print(f"Plataforma detectada: {PLATFORM}")

## Instalação condicional das dependências

Em Kaggle/Colab instala o pacote com os extras geoespaciais e de aprendizado de máquina. No ambiente local a instalação é ignorada, pois é gerenciada por `uv` e pelo CI.

In [ ]:
import subprocess

# Instala o projeto editavelmente com os extras necessários apenas em nuvem.
if PLATFORM in {"kaggle", "colab"}:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[geo,ml]"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("Falha na instalação das dependências:")
        print(result.stderr[-3000:])
        raise RuntimeError("pip install encerrou com erro. Revise a saída acima.")
    print("Dependências instaladas.")
else:
    print("Ambiente local: instalação ignorada (gerenciada por uv/CI).")

## Carregamento da configuração única

Lê a configuração de `src/config.yaml` por meio de `src/config.py`, fonte única de verdade de caminhos, bandas, parâmetros e sementes.

In [ ]:
from src.config import CONFIG

# Exibe um resumo dos parâmetros centrais do estudo.
print(f"Projeto: {CONFIG.get('project.name')} v{CONFIG.get('project.version')}")
print(f"Semente: {CONFIG.seed} | Patch: {CONFIG.patch_size} | Folds: {CONFIG.fold_count}")
print(f"Bandas: {CONFIG.bands}")
print(f"Hash da configuração: {CONFIG.config_hash}")

## Criação da árvore de diretórios

Garante a existência dos diretórios de dados, modelos e artefatos resolvidos a partir da configuração.

In [ ]:
# Cria os diretórios de trabalho, se ainda não existirem.
CONFIG.paths.ensure()
for name in ("data", "raw", "interim", "processed", "external", "models", "artifacts"):
    print(f"{name}: {getattr(CONFIG.paths, name)}")

## Fixação das sementes

Fixa as sementes de `python`, `numpy`, `torch` e `cuda` para garantir a reprodutibilidade dos experimentos.

In [ ]:
from src.config import seed_everything

# Aplica a semente global definida na configuração.
resolved_seed = seed_everything()
print(f"Sementes fixadas em {resolved_seed}.")

## Carregamento de segredos

Em Kaggle/Colab injeta os segredos do cofre da plataforma nas variáveis de ambiente esperadas pelo pacote. Nenhum valor é impresso. No ambiente local, os segredos devem vir de variáveis de ambiente ou do arquivo `.env`.

In [ ]:
# Nomes das variáveis de ambiente consumidas pelas fases seguintes.
SECRET_NAMES = (
    "GEE_SERVICE_ACCOUNT_EMAIL",
    "GEE_PROJECT",
    "GEE_SERVICE_ACCOUNT_KEY_JSON",
    "GEE_OAUTH_CREDENTIALS_JSON",
    "HF_TOKEN",
    "HF_USERNAME",
)

if PLATFORM == "kaggle":
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    for name in SECRET_NAMES:
        try:
            os.environ[name] = client.get_secret(name)
        except Exception:
            print(f"Segredo ausente no Kaggle: {name}")
elif PLATFORM == "colab":
    from google.colab import userdata

    for name in SECRET_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            print(f"Segredo ausente no Colab: {name}")
else:
    print("Ambiente local: segredos esperados via variáveis de ambiente/.env.")

## Autenticação no Google Earth Engine

Inicializa o Earth Engine com as credenciais lidas exclusivamente do ambiente: conta de serviço ou credenciais OAuth de usuário. A ausência de credenciais é reportada sem interromper a execução do notebook.

In [ ]:
from src.data.gee_client import GEECredentialsError, init_ee

# Inicializa o cliente do Earth Engine a partir das variáveis de ambiente.
try:
    ee = init_ee()
    GEE_READY = True
    print("Earth Engine autenticado com sucesso.")
except GEECredentialsError as exc:
    GEE_READY = False
    print(f"Credenciais do GEE ausentes: {exc}")

## Registro do ambiente do run

Persiste versões de dependências, plataforma, commit, hash da configuração e semente em `artifacts/environment.json`, sem expor credenciais.

In [ ]:
from src.config import log_environment, save_environment_log

# Salva o retrato do ambiente e exibe o conteúdo registrado.
log_path = save_environment_log()
print(f"Registro do ambiente salvo em: {log_path}")
log_environment()